# 43. 프로젝트 A — 사내 문서 QA 시스템

> **제43장** · **이론편 대응: 20~23장 종합**
> **예상 소요**: 120분
> **필요 사양**: **[CPU]** 로 실행 가능
> **선행 장**: **32번을 먼저 실행** (utils 모듈 생성)
> **API 키**: 선택 (없어도 검색까지 전부 동작)

---

## 만들 것

**사내 규정을 검색해 근거와 함께 답하는 시스템**

```
질문 → 검색 → 근거 선별 → 답변 생성 → 출처 표시 → 품질 측정
```

| 단계 | 절 | 쓰는 기술 |
|---|---|---|
| 1 | 문서 준비와 분할 | 27장 7절 |
| 2 | 검색기 구축 | 27장 6절 |
| 3 | **평가 데이터 만들기** ★ | 27장 8절 |
| 4 | 기준 성능 측정 | 27장 8절 |
| 5 | **검색 개선** ★ | 27장, 23번 |
| 6 | 답변 생성 | 20·21·23번 |
| 7 | 전체 시스템 조립 | — |
| 8 | 운영 지표 | 31번 |

**3~5절이 이 프로젝트의 핵심이다.**
"측정 → 개선 → 재측정"이라는 실무의 기본 절차를 그대로 따른다.

In [ ]:
import sys
from pathlib import Path

# 42장에서 만든 utils 사용
root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

try:
    from utils import set_seed, setup_korean_font, cosine_similarity
    setup_korean_font()
    set_seed(42)
    print("utils 모듈 로드 완료 (42장에서 생성)")
except ImportError:
    print("[주의] utils 모듈이 없습니다. 42장을 먼저 실행하세요.")
    print("       여기서는 최소 설정으로 진행합니다.")
    import matplotlib.pyplot as plt
    import matplotlib.font_manager as fm
    import platform
    import numpy as np
    _c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
          "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
    _a = {f.name for f in fm.fontManager.ttflist}
    for _n in _c.get(platform.system(), []):
        if _n in _a:
            plt.rcParams["font.family"] = _n
            break
    plt.rcParams["axes.unicode_minus"] = False
    def cosine_similarity(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

import numpy as np
import matplotlib.pyplot as plt
import time
import os
import json

# API 키 (21번 방식)
try:
    from dotenv import load_dotenv
    load_dotenv(root / ".env")
except ImportError:
    pass

API_KEY, BASE_URL, MODEL = None, None, "gpt-4o-mini"
for env_name, base, model in [
        ("OPENAI_API_KEY", None, "gpt-4o-mini"),
        ("GROQ_API_KEY", "https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
        ("GEMINI_API_KEY", "https://generativelanguage.googleapis.com/v1beta/openai/", "gemini-2.0-flash")]:
    if os.getenv(env_name):
        API_KEY, BASE_URL, MODEL = os.getenv(env_name), base, model
        print(f"API 키: {env_name} ({MODEL})")
        break
if not API_KEY:
    print("API 키 없음 — 검색 부분은 전부 동작합니다 (1~5절, 8절)")

---

## 1. 문서 준비와 분할 — 27장 7절

실제 사내 문서를 흉내 낸 데이터를 만든다. **일부러 길게** 만들어 분할이 필요한 상황을 만든다.

In [ ]:
# 사내 규정 문서 (원문 형태 — 문단이 여럿)
raw_documents = {
    "재택근무 규정": """재택근무는 주 2회까지 신청할 수 있습니다. 신청은 최소 3일 전에
사내 포털을 통해 하며, 팀장 승인이 필요합니다.

재택근무일에도 코어타임(오전 10시~오후 4시)에는 연락이 가능해야 합니다.
업무 시작과 종료 시 팀 채널에 보고합니다.

재택근무 중 발생한 통신비는 별도 지원되지 않습니다. 다만 장기 재택이
필요한 경우 별도 협의가 가능합니다.""",

    "연차 규정": """연차는 입사 1년 미만 직원의 경우 매월 1일씩 발생합니다.
입사 1년 이상이면 연 15일이 일괄 부여됩니다.

연차 신청은 최소 2일 전에 해야 하며, 3일 이상 연속 사용 시 1주일 전
신청이 필요합니다. 미사용 연차는 다음 해로 이월되지 않습니다.

경조사 휴가는 연차와 별도로 부여됩니다. 결혼 5일, 배우자 출산 10일,
직계가족 사망 5일입니다.""",

    "출장비 규정": """국내 출장비는 하루 8만원까지 정산 가능합니다. 숙박비는
실비 정산이며 1박당 10만원을 상한으로 합니다.

해외 출장비는 하루 15만원까지이며, 지역에 따라 조정될 수 있습니다.
항공료는 이코노미석을 기준으로 하되, 6시간 이상 비행은 프리미엄
이코노미가 허용됩니다.

모든 출장비는 영수증 첨부가 필수입니다. 출장 종료 후 7일 이내에
정산 신청을 해야 합니다.""",

    "교육비 규정": """교육비는 연간 200만원까지 지원됩니다. 업무 관련성이
인정되어야 하며, 부서장 승인이 필요합니다.

지원 대상은 외부 교육기관 수강료, 온라인 강의, 도서 구입비,
자격증 응시료입니다. 어학 교육은 연 100만원까지로 제한됩니다.

교육 수료 후 수료증을 제출해야 하며, 수료하지 못한 경우 지원금을
반납해야 합니다.""",

    "장비 규정": """업무용 장비는 사내 포털에서 신청합니다. 노트북은 3년마다
교체 대상이 되며, 고장 시에는 즉시 교체 신청이 가능합니다.

모니터, 키보드, 마우스 등 주변기기는 필요 시 신청할 수 있습니다.
개인 선호에 따른 사양 변경은 차액을 본인이 부담합니다.

퇴사 시 모든 장비를 반납해야 합니다. 분실이나 고의 파손은 변상
책임이 있습니다.""",
}

print("=" * 70)
print("원본 문서")
print("=" * 70)
for name, text in raw_documents.items():
    n_para = len([p for p in text.split("\n\n") if p.strip()])
    print(f"  {name:<16}{len(text):>5}자, {n_para}개 문단")

total_chars = sum(len(t) for t in raw_documents.values())
print("-" * 70)
print(f"  {'합계':<16}{total_chars:>5}자")

In [ ]:
def split_documents(docs, min_chars=40):
    """문단 단위로 나눈다 (27장 7절)

    각 조각에 어느 문서에서 왔는지 기록해 둔다 → 출처 표시에 쓴다.
    """
    chunks = []
    for doc_name, text in docs.items():
        paragraphs = [p.strip().replace("\n", " ")
                      for p in text.split("\n\n") if p.strip()]
        for i, para in enumerate(paragraphs):
            if len(para) < min_chars:
                continue
            chunks.append({
                "id": f"{doc_name}#{i}",
                "source": doc_name,
                "para_index": i,
                "text": para,
            })
    return chunks


chunks = split_documents(raw_documents)

print("=" * 78)
print("문단 단위 분할")
print("=" * 78)
print(f"문서 {len(raw_documents)}개 → 조각 {len(chunks)}개")
print()
print(f"{'ID':<20}{'글자수':<10}{'내용'}")
print("-" * 78)
for c in chunks[:8]:
    print(f"{c['id']:<20}{len(c['text']):<10}{c['text'][:38]}...")
print(f"... (총 {len(chunks)}개)")
print("-" * 78)
print()
lengths = [len(c["text"]) for c in chunks]
print(f"조각 길이: 최소 {min(lengths)}자, 최대 {max(lengths)}자, 평균 {np.mean(lengths):.0f}자")
print()
print("[왜 문단 단위인가]")
print("  27장 7절에서 봤듯 통째로 넣으면 여러 주제가 섞여 검색이 부정확해진다.")
print("  문단은 대체로 하나의 주제를 담고 있어 자연스러운 경계가 된다.")

---

## 2. 검색기 구축 — 27장 6절

27장에서 만든 `SimpleVectorStore`를 **출처 정보까지 담도록** 확장한다.

In [ ]:
import numpy as np
import time
from sentence_transformers import SentenceTransformer

print("임베딩 모델 로드 중... (27장에서 받았다면 즉시)")
t0 = time.time()
embedder = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
print(f"완료: {time.time()-t0:.1f}초, 차원 {embedder.get_sentence_embedding_dimension()}")
print()
print("27장 5절에서 확인했듯 한국어에는 다국어 모델이 필수다.")


class DocumentSearcher:
    # 출처 정보를 함께 관리하는 검색기 (27장 6절 확장)

    def __init__(self, embedder):
        self.embedder = embedder
        self.chunks = []
        self.embeddings = None

    def index(self, chunks, verbose=True):
        """조각들을 벡터로 만들어 저장"""
        t0 = time.time()
        texts = [c["text"] for c in chunks]
        self.embeddings = self.embedder.encode(
            texts, normalize_embeddings=True, show_progress_bar=False)
        self.chunks = chunks
        if verbose:
            print(f"인덱싱 완료: {len(chunks)}개 ({time.time()-t0:.1f}초)")
            print(f"  벡터 크기: {self.embeddings.shape}")
            print(f"  메모리: {self.embeddings.nbytes/1024:.1f} KB")

    def search(self, query, top_k=3, min_score=None):
        """검색 — 점수와 출처를 함께 돌려준다"""
        q_emb = self.embedder.encode([query], normalize_embeddings=True)[0]
        scores = self.embeddings @ q_emb        # 정규화했으므로 내적 = 코사인

        order = np.argsort(scores)[::-1]
        results = []
        for i in order[:top_k]:
            score = float(scores[i])
            if min_score is not None and score < min_score:
                continue
            results.append({**self.chunks[i], "score": score})
        return results


searcher = DocumentSearcher(embedder)
searcher.index(chunks)

print()
print("=" * 78)
print("검색 시험")
print("=" * 78)
for q in ["재택근무 신청 방법", "연차 이월", "해외 출장 항공료"]:
    print(f"\n질문: {q}")
    for r in searcher.search(q, top_k=2):
        print(f"  [{r['score']:.4f}] {r['source']} — {r['text'][:38]}...")

---

## 3. 평가 데이터 만들기 ★ — 27장 8절

**개선하려면 먼저 측정해야 한다.** 측정하려면 정답이 있는 질문 목록이 필요하다.

이것이 실무에서 가장 자주 건너뛰는 단계이면서, **가장 중요한 단계**다.
평가 데이터 없이는 "좋아졌는지" 알 수 없기 때문이다.

In [ ]:
# 평가 데이터 — 질문과 정답 조각 ID
# 실제로는 사용자 질문 로그에서 뽑거나 도메인 전문가가 만든다
eval_set = [
    # 쉬운 질문 — 키워드가 문서에 그대로 있음
    {"q": "재택근무는 주 몇 회까지 가능한가요?",     "gold": "재택근무 규정#0"},
    {"q": "연차는 며칠 부여되나요?",                "gold": "연차 규정#0"},
    {"q": "국내 출장비 한도는 얼마인가요?",          "gold": "출장비 규정#0"},
    {"q": "교육비는 얼마까지 지원되나요?",           "gold": "교육비 규정#0"},
    {"q": "노트북은 몇 년마다 교체되나요?",          "gold": "장비 규정#0"},

    # 중간 — 표현이 다름
    {"q": "집에서 일하려면 어떻게 신청하나요?",       "gold": "재택근무 규정#0"},
    {"q": "안 쓴 휴가는 내년에 쓸 수 있나요?",       "gold": "연차 규정#1"},
    {"q": "비행기 좌석 등급 규정",                  "gold": "출장비 규정#1"},

    # 어려움 — 세부 조항
    {"q": "결혼하면 며칠 쉬나요?",                  "gold": "연차 규정#2"},
    {"q": "어학 공부도 지원되나요?",                "gold": "교육비 규정#1"},
    {"q": "퇴사할 때 장비는 어떻게 하나요?",         "gold": "장비 규정#2"},
    {"q": "재택근무 중에도 연락이 되어야 하나요?",    "gold": "재택근무 규정#1"},
]

print("=" * 78)
print("평가 데이터")
print("=" * 78)
print(f"질문 {len(eval_set)}개, 문서 조각 {len(chunks)}개")
print()
print(f"{'질문':<34}{'정답 조각'}")
print("-" * 78)
for e in eval_set:
    print(f"{e['q']:<34}{e['gold']}")
print("-" * 78)
print()

# 정답 ID가 실제로 존재하는지 확인
chunk_ids = {c["id"] for c in chunks}
missing = [e["gold"] for e in eval_set if e["gold"] not in chunk_ids]
if missing:
    print(f"[오류] 존재하지 않는 정답 ID: {missing}")
else:
    print("[OK] 모든 정답 ID가 실제 조각에 존재한다")
print()
print("[평가 데이터를 만들 때의 요령]")
print("  1) 난이도를 섞는다 — 쉬운 것만 있으면 개선 여지가 안 보인다")
print("  2) 실제 사용자가 쓸 법한 표현으로 — '집에서 일하려면' 같은")
print("  3) 최소 20~50개는 있어야 통계적으로 의미가 있다 (여기서는 12개)")

---

## 4. 기준 성능 측정 — 27장 8절

**개선하기 전의 성능**을 먼저 재 둔다. 이것이 비교 기준(baseline)이 된다.

In [ ]:
import numpy as np


def evaluate_search(searcher, eval_set, k_values=(1, 3, 5), verbose=False):
    """검색 성능 평가 (27장 8절)"""
    max_k = max(k_values)
    hits = {k: 0 for k in k_values}
    reciprocal_ranks = []
    details = []

    for e in eval_set:
        results = searcher.search(e["q"], top_k=max_k)
        ids = [r["id"] for r in results]

        if e["gold"] in ids:
            rank = ids.index(e["gold"]) + 1
            reciprocal_ranks.append(1.0 / rank)
            for k in k_values:
                if rank <= k:
                    hits[k] += 1
        else:
            rank = None
            reciprocal_ranks.append(0.0)

        details.append({
            "q": e["q"], "gold": e["gold"], "rank": rank,
            "top1": ids[0] if ids else None,
            "top1_score": results[0]["score"] if results else 0.0,
        })

    n = len(eval_set)
    return {
        "recall": {k: hits[k] / n for k in k_values},
        "mrr": float(np.mean(reciprocal_ranks)),
        "details": details,
    }


baseline = evaluate_search(searcher, eval_set)

print("=" * 78)
print("기준 성능 (baseline)")
print("=" * 78)
print(f"{'지표':<16}{'값':<14}{'뜻'}")
print("-" * 78)
for k, v in baseline["recall"].items():
    print(f"{'Recall@'+str(k):<16}{v:<14.4f}상위 {k}개 안에 정답이 있는 비율")
print(f"{'MRR':<16}{baseline['mrr']:<14.4f}정답의 평균 역순위")
print("-" * 78)
print()

print("질문별 상세")
print(f"{'질문':<34}{'정답 순위':<12}{'1위 조각':<20}{'점수'}")
print("-" * 78)
for d in baseline["details"]:
    rank = str(d["rank"]) if d["rank"] else "없음"
    mark = "" if d["rank"] == 1 else "  ←"
    print(f"{d['q'][:32]:<34}{rank:<12}{str(d['top1'])[:18]:<20}{d['top1_score']:.3f}{mark}")
print("-" * 78)
print()
failed = [d for d in baseline["details"] if d["rank"] != 1]
print(f"1위로 못 찾은 질문: {len(failed)}개")
for d in failed:
    print(f"  '{d['q']}'")
    print(f"    기대: {d['gold']}   실제 1위: {d['top1']}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

ax = axes[0]
ks = list(baseline["recall"].keys())
vals = list(baseline["recall"].values())
bars = ax.bar([f"@{k}" for k in ks], vals, color="#1E40AF")
for b, v in zip(bars, vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.02, f"{v:.2f}", ha="center", fontsize=10)
ax.set_ylabel("Recall")
ax.set_title("기준 성능 — Recall@k")
ax.set_ylim(0, 1.1)
ax.grid(axis="y", alpha=0.3)

ax = axes[1]
ranks = [d["rank"] if d["rank"] else 99 for d in baseline["details"]]
rank_counts = {}
for r in ranks:
    key = str(r) if r <= 5 else "6위 이하/없음"
    rank_counts[key] = rank_counts.get(key, 0) + 1
ax.bar(list(rank_counts.keys()), list(rank_counts.values()), color="#0D9488")
ax.set_xlabel("정답의 순위")
ax.set_ylabel("질문 수")
ax.set_title("정답이 몇 위에 있었나")
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("이 수치가 개선의 출발점이다.")
print("  무엇을 바꾸든 이 값과 비교해 좋아졌는지 판단한다.")

---

## 5. 검색 개선 ★ — 27장, 28번

기준을 잡았으니 **개선을 시도**한다. 세 가지 방법을 순서대로 적용하고
**각각이 얼마나 효과가 있는지** 측정한다.

| 방법 | 아이디어 | 22·28장 참조 |
|---|---|---|
| 1. 문서 제목 포함 | 조각에 출처 정보를 함께 임베딩 | — |
| 2. 하이브리드 검색 | 키워드 검색과 결합 | 27장 4절 |
| 3. 질의 확장 | 질문을 검색에 맞게 다듬기 | 28장 8절 |

In [ ]:
import numpy as np

print("=" * 78)
print("개선 1: 조각에 문서 제목 포함")
print("=" * 78)
print()
print("아이디어")
print("  '연차 규정' 이라는 제목 자체가 검색에 도움이 되는 정보다.")
print("  조각 본문에만 의존하면 이 정보가 버려진다.")
print()

# 제목을 붙인 조각 만들기
chunks_with_title = [
    {**c, "text": f"[{c['source']}] {c['text']}"}
    for c in chunks
]

searcher_v2 = DocumentSearcher(embedder)
searcher_v2.index(chunks_with_title, verbose=False)
result_v2 = evaluate_search(searcher_v2, eval_set)

print(f"{'지표':<16}{'기준':<14}{'제목 포함':<14}{'변화'}")
print("-" * 78)
for k in [1, 3, 5]:
    b = baseline["recall"][k]
    v = result_v2["recall"][k]
    print(f"{'Recall@'+str(k):<16}{b:<14.4f}{v:<14.4f}{v-b:+.4f}")
print(f"{'MRR':<16}{baseline['mrr']:<14.4f}{result_v2['mrr']:<14.4f}"
      f"{result_v2['mrr']-baseline['mrr']:+.4f}")
print("-" * 78)
print()
if result_v2["mrr"] > baseline["mrr"]:
    print("[개선됨] 제목 정보가 검색에 도움이 되었다.")
elif result_v2["mrr"] < baseline["mrr"]:
    print("[악화됨] 제목이 오히려 방해가 되었다.")
    print("  모든 조각에 같은 제목이 붙으면 조각 간 구별이 흐려질 수 있다.")
else:
    print("[변화 없음]")

In [ ]:
import re
import numpy as np


class HybridSearcher(DocumentSearcher):
    # 의미 검색 + 키워드 검색 (27장 4절)

    def _keyword_scores(self, query):
        """단순 키워드 겹침 점수"""
        q_tokens = set(re.findall(r"[가-힣a-zA-Z]+", query.lower()))
        scores = []
        for c in self.chunks:
            d_tokens = set(re.findall(r"[가-힣a-zA-Z]+", c["text"].lower()))
            if not q_tokens:
                scores.append(0.0)
                continue
            overlap = len(q_tokens & d_tokens)
            scores.append(overlap / len(q_tokens))
        return np.array(scores)

    def search(self, query, top_k=3, alpha=0.7, min_score=None):
        """alpha: 의미 검색 가중치 (1.0이면 의미만, 0.0이면 키워드만)"""
        q_emb = self.embedder.encode([query], normalize_embeddings=True)[0]
        semantic = self.embeddings @ q_emb
        keyword = self._keyword_scores(query)

        # 두 점수의 범위가 다르므로 각각 정규화 후 결합
        def norm(x):
            rng = x.max() - x.min()
            return (x - x.min()) / rng if rng > 1e-9 else np.zeros_like(x)

        combined = alpha * norm(semantic) + (1 - alpha) * norm(keyword)

        order = np.argsort(combined)[::-1]
        results = []
        for i in order[:top_k]:
            results.append({**self.chunks[i],
                            "score": float(combined[i]),
                            "semantic": float(semantic[i]),
                            "keyword": float(keyword[i])})
        return results


print("=" * 78)
print("개선 2: 하이브리드 검색 (27장 4절)")
print("=" * 78)
print()

hybrid = HybridSearcher(embedder)
hybrid.index(chunks, verbose=False)

print(f"{'alpha':<12}{'설명':<24}{'Recall@1':<14}{'MRR'}")
print("-" * 78)

best_alpha, best_mrr = None, -1
for alpha in [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]:
    # alpha 를 고정한 검색기로 평가
    class _Tuned(HybridSearcher):
        def search(self, q, top_k=3, min_score=None):
            return HybridSearcher.search(self, q, top_k=top_k, alpha=alpha)

    tuned = _Tuned(embedder)
    tuned.chunks = hybrid.chunks
    tuned.embeddings = hybrid.embeddings
    r = evaluate_search(tuned, eval_set)

    desc = {0.0: "키워드만", 1.0: "의미만"}.get(alpha, "혼합")
    print(f"{alpha:<12}{desc:<24}{r['recall'][1]:<14.4f}{r['mrr']:.4f}")

    if r["mrr"] > best_mrr:
        best_mrr, best_alpha = r["mrr"], alpha

print("-" * 78)
print(f"최적 alpha: {best_alpha} (MRR {best_mrr:.4f})")
print(f"기준 대비: {best_mrr - baseline['mrr']:+.4f}")
print()
print("27장 4절에서 다룬 대로 두 방식은 강점이 다르다.")
print("  키워드: 정확한 용어가 있을 때")
print("  의미  : 표현이 다를 때")

In [ ]:
print("=" * 78)
print("개선 3: 질의 확장 (28장 8절)")
print("=" * 78)
print()
print("아이디어")
print("  '집에서 일하려면' 같은 구어체 질문은 문서 표현과 멀다.")
print("  검색에 적합한 형태로 다듬으면 찾기 쉬워진다.")
print()

# 규칙 기반 동의어 확장 (실무에서는 LLM 을 쓰기도 한다)
SYNONYMS = {
    "집에서 일": "재택근무",
    "재택": "재택근무",
    "휴가": "연차",
    "쉬는": "휴가 연차",
    "비행기": "항공료",
    "좌석": "항공료 이코노미",
    "어학": "어학 교육",
    "퇴사": "반납",
    "결혼": "경조사 결혼",
}


def expand_query(query):
    """질문에 관련 용어를 덧붙인다"""
    additions = []
    for key, value in SYNONYMS.items():
        if key in query:
            additions.append(value)
    if additions:
        return query + " " + " ".join(additions)
    return query


print("확장 예시")
print(f"{'원본':<34}{'확장 후'}")
print("-" * 78)
for e in eval_set[:6]:
    expanded = expand_query(e["q"])
    changed = "  ←" if expanded != e["q"] else ""
    print(f"{e['q'][:32]:<34}{expanded[:40]}{changed}")
print("-" * 78)
print()


class ExpandedSearcher(HybridSearcher):
    # 질의 확장 + 하이브리드
    def search(self, query, top_k=3, alpha=0.7, min_score=None):
        expanded = expand_query(query)
        return HybridSearcher.search(self, expanded, top_k=top_k, alpha=alpha)


expanded_searcher = ExpandedSearcher(embedder)
expanded_searcher.chunks = hybrid.chunks
expanded_searcher.embeddings = hybrid.embeddings

result_v4 = evaluate_search(expanded_searcher, eval_set)

print(f"{'지표':<16}{'기준':<14}{'확장 후':<14}{'변화'}")
print("-" * 78)
for k in [1, 3, 5]:
    b = baseline["recall"][k]
    v = result_v4["recall"][k]
    print(f"{'Recall@'+str(k):<16}{b:<14.4f}{v:<14.4f}{v-b:+.4f}")
print(f"{'MRR':<16}{baseline['mrr']:<14.4f}{result_v4['mrr']:<14.4f}"
      f"{result_v4['mrr']-baseline['mrr']:+.4f}")
print("-" * 78)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

print("=" * 78)
print("개선 결과 종합")
print("=" * 78)

variants = {
    "기준 (의미 검색)":      baseline,
    "제목 포함":            result_v2,
    "하이브리드":           evaluate_search(hybrid, eval_set),
    "확장 + 하이브리드":     result_v4,
}

print(f"{'구성':<24}{'Recall@1':<14}{'Recall@3':<14}{'MRR'}")
print("-" * 78)
for name, r in variants.items():
    print(f"{name:<24}{r['recall'][1]:<14.4f}{r['recall'][3]:<14.4f}{r['mrr']:.4f}")
print("-" * 78)

best_name = max(variants, key=lambda k: variants[k]["mrr"])
print(f"최고 성능: {best_name} (MRR {variants[best_name]['mrr']:.4f})")
print(f"기준 대비: {variants[best_name]['mrr'] - baseline['mrr']:+.4f}")

fig, ax = plt.subplots(figsize=(9.5, 4.5))
names = list(variants.keys())
x = np.arange(len(names))
width = 0.28

ax.bar(x - width, [variants[n]["recall"][1] for n in names], width,
       label="Recall@1", color="#1E40AF")
ax.bar(x, [variants[n]["recall"][3] for n in names], width,
       label="Recall@3", color="#0D9488")
ax.bar(x + width, [variants[n]["mrr"] for n in names], width,
       label="MRR", color="#EA580C")

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=8, rotation=12, ha="right")
ax.set_ylabel("점수")
ax.set_title("검색 개선 결과")
ax.set_ylim(0, 1.15)
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print()
print("[이 과정에서 배울 것]")
print("  모든 개선이 효과가 있는 것은 아니다.")
print("  측정해 보지 않으면 어느 것이 도움이 됐는지 알 수 없다.")
print("  때로는 개선이 오히려 성능을 떨어뜨린다.")

---

## 6. 답변 생성 — 20·21·28번

검색이 정리됐으니 **답변을 만든다.** 28장 6절에서 다듬은 프롬프트를 쓴다.

In [ ]:
import json


def call_llm(messages, max_tokens=400, temperature=0.2):
    """LLM 호출 (21번 방식)"""
    if not API_KEY:
        return None
    try:
        from openai import OpenAI
        kwargs = {"api_key": API_KEY}
        if BASE_URL:
            kwargs["base_url"] = BASE_URL
        client = OpenAI(**kwargs)
        r = client.chat.completions.create(
            model=MODEL, messages=messages,
            max_tokens=max_tokens, temperature=temperature)
        return r.choices[0].message.content
    except Exception as e:
        print(f"[오류] {type(e).__name__}: {str(e)[:120]}")
        return None


SYSTEM_PROMPT = """당신은 사내 규정을 안내하는 도우미입니다.

규칙:
1. 아래 제공된 문서만을 근거로 답하세요.
2. 답변에 사용한 문서 번호를 [1], [2] 형식으로 표시하세요.
3. 문서에 없는 내용은 "제공된 문서에서 찾을 수 없습니다"라고 답하세요.
4. 추측하거나 일반 상식으로 보충하지 마세요.
5. 간결하게 답하되, 조건이나 예외가 있으면 함께 알려주세요."""


def build_prompt(question, docs):
    """검색 결과로 프롬프트를 만든다 (28장 6절)"""
    context = "\n\n".join(
        f"[{i+1}] ({d['source']}) {d['text']}"
        for i, d in enumerate(docs))
    user = f"[참고 문서]\n{context}\n\n[질문]\n{question}"
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user}]


print("=" * 78)
print("프롬프트 구성 확인")
print("=" * 78)

question = "재택근무 신청은 어떻게 하나요?"
docs = expanded_searcher.search(question, top_k=3)

messages = build_prompt(question, docs)
print(messages[1]["content"][:600])
print("...")
print()
print(f"프롬프트 길이: system {len(messages[0]['content'])}자 + "
      f"user {len(messages[1]['content'])}자")

In [ ]:
print("=" * 78)
print("답변 생성")
print("=" * 78)

test_questions = [
    "재택근무 신청은 어떻게 하나요?",
    "연차를 3일 연속 쓰려면 언제 신청해야 하나요?",
    "주차비도 지원되나요?",          # 문서에 없는 내용
]

for q in test_questions:
    print(f"\n{'='*78}")
    print(f"질문: {q}")
    print("-" * 78)

    docs = expanded_searcher.search(q, top_k=3)
    print("검색된 근거")
    for i, d in enumerate(docs, 1):
        print(f"  [{i}] ({d['score']:.3f}) {d['source']} — {d['text'][:40]}...")

    answer = call_llm(build_prompt(q, docs))
    print()
    if answer:
        print("답변")
        for line in answer.strip().split("\n"):
            print(f"  {line}")
    else:
        print("답변: (API 키 없음 — 검색까지는 정상 동작)")

if not API_KEY:
    print()
    print("=" * 78)
    print("API 키가 있으면 여기서 근거 기반 답변이 생성된다.")
    print("  마지막 질문('주차비')은 문서에 없으므로")
    print("  '제공된 문서에서 찾을 수 없습니다'가 나와야 정상이다.")

---

## 7. 전체 시스템 조립

지금까지 만든 것을 **하나의 클래스로** 묶는다.

In [ ]:
import time
import numpy as np


class DocumentQASystem:
    # 사내 문서 QA 시스템 (전체 조립)

    def __init__(self, searcher, top_k=3, min_score=0.25):
        self.searcher = searcher
        self.top_k = top_k
        self.min_score = min_score      # 이보다 낮으면 "모른다"고 답한다
        self.logs = []

    def answer(self, question, verbose=True):
        t0 = time.time()

        # 1) 검색
        docs = self.searcher.search(question, top_k=self.top_k)
        search_time = time.time() - t0

        # 2) 신뢰도 확인 — 너무 낮으면 생성하지 않는다
        if not docs or docs[0]["score"] < self.min_score:
            result = {
                "question": question,
                "answer": "관련 규정을 찾을 수 없습니다. 담당 부서에 문의해 주세요.",
                "sources": [],
                "confidence": docs[0]["score"] if docs else 0.0,
                "search_time": search_time,
                "total_time": time.time() - t0,
                "generated": False,
            }
            self.logs.append(result)
            if verbose:
                self._print(result)
            return result

        # 3) 답변 생성
        answer = call_llm(build_prompt(question, docs))

        result = {
            "question": question,
            "answer": answer or "(API 키 없음 — 아래 근거를 참고하세요)",
            "sources": [{"id": d["id"], "source": d["source"],
                         "score": d["score"], "text": d["text"]} for d in docs],
            "confidence": docs[0]["score"],
            "search_time": search_time,
            "total_time": time.time() - t0,
            "generated": answer is not None,
        }
        self.logs.append(result)
        if verbose:
            self._print(result)
        return result

    def _print(self, r):
        print(f"\n질문: {r['question']}")
        print(f"신뢰도: {r['confidence']:.3f}")
        print()
        print("답변")
        for line in str(r["answer"]).strip().split("\n"):
            print(f"  {line}")
        if r["sources"]:
            print()
            print("근거")
            for i, s in enumerate(r["sources"], 1):
                print(f"  [{i}] ({s['score']:.3f}) {s['source']}")
                print(f"      {s['text'][:60]}...")
        print(f"\n소요: 검색 {r['search_time']*1000:.0f}ms / "
              f"전체 {r['total_time']*1000:.0f}ms")


qa = DocumentQASystem(expanded_searcher, top_k=3, min_score=0.25)

print("=" * 78)
print("전체 시스템 실행")
print("=" * 78)
qa.answer("교육비로 어학 공부를 할 수 있나요?")

In [ ]:
print("=" * 78)
print("여러 질문 처리")
print("=" * 78)

queries = [
    "출장 다녀오면 언제까지 정산해야 하나요?",
    "노트북이 고장났는데 어떻게 하나요?",
    "회사 주차장 이용 방법",          # 문서에 없음
]

for q in queries:
    qa.answer(q)
    print("\n" + "="*78)

print()
print(f"처리한 질문: {len(qa.logs)}개")
print()
print("[min_score 의 역할]")
print("  검색 점수가 낮으면 답변을 생성하지 않고 '모른다'고 한다.")
print("  28장 5절에서 봤듯 엉뚱한 문서로 답하면 잘못된 답이 나오기 때문이다.")
print()
print("  이 임계값을 정하는 것도 측정으로 한다 — 다음 절에서 다룬다.")

---

## 8. 운영 지표 — 41번

실제로 운영한다면 **무엇을 봐야 하는지** 정리한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("min_score 임계값 정하기")
print("=" * 78)
print()
print("너무 낮으면: 엉뚱한 문서로 답한다")
print("너무 높으면: 답할 수 있는데도 '모른다'고 한다")
print()

# 평가 데이터로 점수 분포 확인
correct_scores, wrong_scores = [], []
for e in eval_set:
    results = expanded_searcher.search(e["q"], top_k=1)
    if results:
        if results[0]["id"] == e["gold"]:
            correct_scores.append(results[0]["score"])
        else:
            wrong_scores.append(results[0]["score"])

# 문서에 없는 질문 (거절해야 하는 것)
oov_questions = ["주차장 이용 방법", "구내식당 메뉴", "헬스장 할인"]
oov_scores = [expanded_searcher.search(q, top_k=1)[0]["score"]
              for q in oov_questions]

print(f"{'구분':<24}{'개수':<10}{'평균 점수':<14}{'최소':<10}{'최대'}")
print("-" * 78)
for name, scores in [("정답을 1위로 찾음", correct_scores),
                     ("틀린 문서가 1위", wrong_scores),
                     ("문서에 없는 질문", oov_scores)]:
    if scores:
        print(f"{name:<24}{len(scores):<10}{np.mean(scores):<14.4f}"
              f"{min(scores):<10.4f}{max(scores):.4f}")
print("-" * 78)
print()

if correct_scores and oov_scores:
    gap_low = max(oov_scores)
    gap_high = min(correct_scores)
    print(f"문서에 없는 질문의 최고 점수: {gap_low:.4f}")
    print(f"정답을 찾은 경우의 최저 점수: {gap_high:.4f}")
    if gap_high > gap_low:
        suggested = (gap_low + gap_high) / 2
        print(f"→ 임계값 후보: {suggested:.4f}")
    else:
        print("→ 두 분포가 겹친다. 임계값만으로는 완전히 나눌 수 없다.")
        print("   07장에서 다룬 정밀도-재현율 맞바꿈과 같은 상황이다.")

fig, ax = plt.subplots(figsize=(9, 4))
if correct_scores:
    ax.hist(correct_scores, bins=10, alpha=0.6, label="정답 찾음", color="#0D9488")
if wrong_scores:
    ax.hist(wrong_scores, bins=10, alpha=0.6, label="틀림", color="#EA580C")
if oov_scores:
    ax.hist(oov_scores, bins=10, alpha=0.6, label="문서에 없음", color="#DC2626")
ax.set_xlabel("검색 점수")
ax.set_ylabel("빈도")
ax.set_title("점수 분포 — 임계값 정하기")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

print("=" * 78)
print("운영 지표 요약")
print("=" * 78)

if qa.logs:
    search_times = [l["search_time"] * 1000 for l in qa.logs]
    total_times = [l["total_time"] * 1000 for l in qa.logs]
    confidences = [l["confidence"] for l in qa.logs]
    refused = sum(1 for l in qa.logs if not l["sources"])

    print(f"{'지표':<24}{'값'}")
    print("-" * 78)
    print(f"{'처리한 질문':<24}{len(qa.logs)}")
    print(f"{'거절한 질문':<24}{refused} ({refused/len(qa.logs)*100:.0f}%)")
    print(f"{'검색 시간 평균':<24}{np.mean(search_times):.1f} ms")
    print(f"{'검색 시간 p95':<24}{np.percentile(search_times, 95):.1f} ms")
    print(f"{'전체 시간 평균':<24}{np.mean(total_times):.1f} ms")
    print(f"{'신뢰도 평균':<24}{np.mean(confidences):.3f}")
    print("-" * 78)

print()
print("실제 운영에서 더 봐야 할 것 (41장 6절)")
print()
print(f"{'지표':<28}{'왜'}")
print("-" * 78)
items = [
    ("거절률", "너무 높으면 문서가 부족하다는 신호"),
    ("재질문률", "같은 것을 다시 물으면 답이 나빴다는 신호"),
    ("근거 클릭률", "사용자가 출처를 확인하는가"),
    ("피드백 (좋아요/싫어요)", "실제 만족도"),
    ("검색 실패 질문 목록", "문서를 보강할 대상"),
]
for a, b in items:
    print(f"{a:<28}{b}")
print("-" * 78)
print()
print("[가장 유용한 것]")
print("  '검색이 실패한 질문 목록'을 모아 두면")
print("  어떤 문서를 추가해야 할지 바로 알 수 있다.")

---

## 9. 개선 과제

이 시스템에는 아직 부족한 점이 많다. **직접 개선해 보자.**

| 난이도 | 과제 | 참고 |
|---|---|---|
| 쉬움 | 평가 질문을 20개 이상으로 늘리기 | 3절 |
| 쉬움 | 문서를 추가하고 재측정 | 1절 |
| 보통 | 조각 크기를 바꿔 비교 (문단 vs 길이 기준) | 27장 7절 |
| 보통 | `top_k`를 바꿔 가며 최적값 찾기 | 28장 5절 |
| 보통 | 답변 품질을 사람이 평가하는 절차 만들기 | 41장 6절 |
| 어려움 | 재순위화 추가 (검색 후보를 다시 정렬) | 27장 8절 |
| 어려움 | 여러 문서에 걸친 질문 처리 | — |
| 어려움 | 대화 맥락 유지 (이전 질문 참조) | 25장 3절 |

**각 개선마다 4절의 평가를 다시 돌려** 정말 좋아졌는지 확인하는 것이 중요하다.

In [ ]:
print("=" * 78)
print("과제 예시: top_k 최적값 찾기")
print("=" * 78)
print()
print("검색 결과를 몇 개나 LLM에게 줄 것인가?")
print()

for k in [1, 2, 3, 5, 8]:
    # 정답이 상위 k개 안에 들어오는 비율
    hits = 0
    total_chars = 0
    for e in eval_set:
        results = expanded_searcher.search(e["q"], top_k=k)
        ids = [r["id"] for r in results]
        if e["gold"] in ids:
            hits += 1
        total_chars += sum(len(r["text"]) for r in results)

    recall = hits / len(eval_set)
    avg_chars = total_chars / len(eval_set)
    print(f"  top_k={k}: Recall {recall:.3f}, 프롬프트 평균 {avg_chars:.0f}자")

print()
print("-" * 78)
print("맞바꿈이 보인다")
print("  k 를 늘리면 정답을 놓칠 확률은 줄지만")
print("  프롬프트가 길어져 비용이 늘고, 관련 없는 문서가 섞인다 (28장 5절)")
print()
print("이 데이터에서는 k=3 정도가 균형점으로 보인다.")
print("  실제로는 답변 품질까지 함께 봐야 정확히 정할 수 있다.")

---

## 10. 정리

### 만든 것

```
문서 → 분할 → 임베딩 → 검색기
                          ↓
질문 → 질의 확장 → 하이브리드 검색 → 신뢰도 확인
                                        ↓
                              프롬프트 조립 → LLM → 답변 + 출처
```

### 이 프로젝트에서 배운 것

| 교훈 | 어디서 |
|---|---|
| **먼저 측정한다** | 3~4절 — 평가 데이터 없이는 개선 불가 |
| 모든 개선이 효과 있는 것은 아니다 | 5절 |
| 검색 품질이 상한을 정한다 | 6절 |
| 모를 때는 모른다고 한다 | 7절 min_score |
| 임계값도 측정으로 정한다 | 8절 |

### 쓴 기술

| 장 | 어디에 |
|---|---|
| 22번 | 임베딩, 코사인 유사도, 하이브리드, 평가 지표 |
| 23번 | RAG 구조, 프롬프트 설계, 출처 표시 |
| 20번 | 생성 파라미터 (temperature 낮게) |
| 21번 | API 호출, 오류 처리 |
| 31번 | 지표 측정, p95 |

### 다음 장

**44. 프로젝트 B — 데이터 분석 Agent**

이 프로젝트가 **"무엇을 아는가"**를 다뤘다면,
다음은 **"무엇을 할 수 있는가"**를 다룬다.

같은 기술 위에 서 있지만 조합이 다르다.